In [ ]:
import tensorflow as tf, numpy as np, matplotlib.pyplot as plt
from scipy import linalg

np.random.seed(42)
tf.random.set_seed(42)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = 10

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train = x_train.astype("float32")
x_train = (x_train - 127.5) / 127.5
y_train = y_train.flatten()

In [ ]:
class Generator(tf.keras.Model):
    def __init__(self, latent_dim=100, embed_dim=50):
        super().__init__()
        self.label_embed = tf.keras.layers.Embedding(num_classes, embed_dim)
        self.flatten = tf.keras.layers.Flatten()
        self.concat = tf.keras.layers.Concatenate()

        self.dense = tf.keras.layers.Dense(4 * 4 * 256)
        self.act0 = tf.keras.layers.LeakyReLU(0.2)
        self.reshape = tf.keras.layers.Reshape((4, 4, 256))

        self.deconv1 = tf.keras.layers.Conv2DTranspose(128, 4, strides=2, padding='same')
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.act1 = tf.keras.layers.LeakyReLU(0.2)

        self.deconv2 = tf.keras.layers.Conv2DTranspose(64, 4, strides=2, padding='same')
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.act2 = tf.keras.layers.LeakyReLU(0.2)

        self.deconv3 = tf.keras.layers.Conv2DTranspose(3, 4, strides=2, padding='same', activation='tanh')

    def call(self, inputs, training=False):
        noise_in, label_in = inputs
        label_embed = self.label_embed(label_in)
        label_embed = self.flatten(label_embed)
        x = self.concat([noise_in, label_embed])

        x = self.dense(x)
        x = self.act0(x)
        x = self.reshape(x)

        x = self.deconv1(x)
        x = self.bn1(x, training=training)
        x = self.act1(x)

        x = self.deconv2(x)
        x = self.bn2(x, training=training)
        x = self.act2(x)

        x = self.deconv3(x)
        return x

In [ ]:
class Discriminator(tf.keras.Model):
    def __init__(self, embed_dim=50):
        super().__init__()
        self.label_embed = tf.keras.layers.Embedding(num_classes, embed_dim)
        self.label_dense = tf.keras.layers.Dense(32 * 32)
        self.label_reshape = tf.keras.layers.Reshape((32, 32, 1))
        self.concat = tf.keras.layers.Concatenate()

        self.conv1 = tf.keras.layers.Conv2D(64, 4, strides=2, padding='same')
        self.act1 = tf.keras.layers.LeakyReLU(0.2)

        self.conv2 = tf.keras.layers.Conv2D(128, 4, strides=2, padding='same')
        self.act2 = tf.keras.layers.LeakyReLU(0.2)

        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(1, activation='sigmoid')

    def call(self, inputs, training=False):
        img_in, label_in = inputs
        label_embed = self.label_embed(label_in)
        label_embed = self.label_dense(label_embed)
        label_embed = self.label_reshape(label_embed)
        x = self.concat([img_in, label_embed])

        x = self.conv1(x)
        x = self.act1(x)

        x = self.conv2(x)
        x = self.act2(x)

        x = self.flatten(x)
        x = self.fc(x)
        return x

In [ ]:
class UnconditionalGenerator(tf.keras.Model):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.dense = tf.keras.layers.Dense(4 * 4 * 256)
        self.act0 = tf.keras.layers.LeakyReLU(0.2)
        self.reshape = tf.keras.layers.Reshape((4, 4, 256))

        self.deconv1 = tf.keras.layers.Conv2DTranspose(128, 4, strides=2, padding='same')
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.act1 = tf.keras.layers.LeakyReLU(0.2)

        self.deconv2 = tf.keras.layers.Conv2DTranspose(64, 4, strides=2, padding='same')
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.act2 = tf.keras.layers.LeakyReLU(0.2)

        self.deconv3 = tf.keras.layers.Conv2DTranspose(3, 4, strides=2, padding='same', activation='tanh')

    def call(self, x, training=False):
        x = self.dense(x)
        x = self.act0(x)
        x = self.reshape(x)

        x = self.deconv1(x)
        x = self.bn1(x, training=training)
        x = self.act1(x)

        x = self.deconv2(x)
        x = self.bn2(x, training=training)
        x = self.act2(x)

        x = self.deconv3(x)
        return x


class UnconditionalDiscriminator(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.conv1 = tf.keras.layers.Conv2D(64, 4, strides=2, padding='same')
        self.act1 = tf.keras.layers.LeakyReLU(0.2)

        self.conv2 = tf.keras.layers.Conv2D(128, 4, strides=2, padding='same')
        self.act2 = tf.keras.layers.LeakyReLU(0.2)

        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(1, activation='sigmoid')

    def call(self, img_in, training=False):
        x = self.conv1(img_in)
        x = self.act1(x)

        x = self.conv2(x)
        x = self.act2(x)

        x = self.flatten(x)
        x = self.fc(x)
        return x

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy()

def train_cgan(gen, disc, images, labels, latent_dim=100, epochs=30,
               batch_size=128, lr=2e-4):
    g_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5, beta_2=0.999)
    d_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5, beta_2=0.999)
    n = images.shape[0]
    n_batches = max(1, n // batch_size)
    g_losses, d_losses = [], []

    for epoch in range(1, epochs + 1):
        idx = np.random.permutation(n)
        eg, ed = [], []
        for b in range(n_batches):
            bi = idx[b * batch_size:(b + 1) * batch_size]
            real_imgs = tf.convert_to_tensor(images[bi], dtype=tf.float32)
            real_labels = tf.convert_to_tensor(labels[bi].reshape(-1, 1), dtype=tf.int32)
            bs = tf.shape(real_imgs)[0]

            noise = tf.random.normal((bs, latent_dim))
            fake_labels = tf.random.uniform((bs, 1), 0, num_classes, dtype=tf.int32)
            with tf.GradientTape() as dt:
                fake_imgs = gen([noise, fake_labels], training=True)
                real_out = disc([real_imgs, real_labels], training=True)
                fake_out = disc([fake_imgs, fake_labels], training=True)
                d_loss = bce(tf.ones_like(real_out), real_out) + bce(tf.zeros_like(fake_out), fake_out)
            dg = dt.gradient(d_loss, disc.trainable_variables)
            d_opt.apply_gradients(zip(dg, disc.trainable_variables))

            noise = tf.random.normal((bs, latent_dim))
            fake_labels = tf.random.uniform((bs, 1), 0, num_classes, dtype=tf.int32)
            with tf.GradientTape() as gt:
                fake_imgs = gen([noise, fake_labels], training=True)
                fake_out = disc([fake_imgs, fake_labels], training=True)
                g_loss = bce(tf.ones_like(fake_out), fake_out)
            gg = gt.gradient(g_loss, gen.trainable_variables)
            g_opt.apply_gradients(zip(gg, gen.trainable_variables))

            eg.append(float(g_loss)); ed.append(float(d_loss))

        g_losses.append(np.mean(eg)); d_losses.append(np.mean(ed))
        print(f"Epoch {epoch}/{epochs}  G_loss={g_losses[-1]:.4f}  D_loss={d_losses[-1]:.4f}")

    return g_losses, d_losses

def train_unconditional_gan(gen, disc, images, latent_dim=100, epochs=30,
                             batch_size=128, lr=2e-4):
    g_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5, beta_2=0.999)
    d_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5, beta_2=0.999)
    n = images.shape[0]
    n_batches = max(1, n // batch_size)
    g_losses, d_losses = [], []

    for epoch in range(1, epochs + 1):
        idx = np.random.permutation(n)
        eg, ed = [], []
        for b in range(n_batches):
            bi = idx[b * batch_size:(b + 1) * batch_size]
            real_imgs = tf.convert_to_tensor(images[bi], dtype=tf.float32)
            bs = tf.shape(real_imgs)[0]

            noise = tf.random.normal((bs, latent_dim))
            with tf.GradientTape() as dt:
                fake_imgs = gen(noise, training=True)
                real_out = disc(real_imgs, training=True)
                fake_out = disc(fake_imgs, training=True)
                d_loss = bce(tf.ones_like(real_out), real_out) + bce(tf.zeros_like(fake_out), fake_out)
            dg = dt.gradient(d_loss, disc.trainable_variables)
            d_opt.apply_gradients(zip(dg, disc.trainable_variables))

            noise = tf.random.normal((bs, latent_dim))
            with tf.GradientTape() as gt:
                fake_imgs = gen(noise, training=True)
                fake_out = disc(fake_imgs, training=True)
                g_loss = bce(tf.ones_like(fake_out), fake_out)
            gg = gt.gradient(g_loss, gen.trainable_variables)
            g_opt.apply_gradients(zip(gg, gen.trainable_variables))

            eg.append(float(g_loss)); ed.append(float(d_loss))

        g_losses.append(np.mean(eg)); d_losses.append(np.mean(ed))
        print(f"[Unconditional] Epoch {epoch}/{epochs}  G_loss={g_losses[-1]:.4f}  D_loss={d_losses[-1]:.4f}")

    return g_losses, d_losses

In [ ]:
def show_images(images, titles=None, suptitle=None, n_cols=5):
    images = (images + 1) / 2.0
    n = images.shape[0]
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2 * n_cols, 2 * n_rows))
    axes = np.array(axes).reshape(-1)
    for i in range(len(axes)):
        axes[i].axis('off')
        if i < n:
            axes[i].imshow(np.clip(images[i], 0, 1))
            if titles is not None:
                axes[i].set_title(titles[i], fontsize=9)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()

In [ ]:
gen = Generator(latent_dim=100, embed_dim=50)
disc = Discriminator(embed_dim=50)
g_losses, d_losses = train_cgan(gen, disc, x_train, y_train, latent_dim=100, epochs=10, batch_size=128)

In [ ]:
plt.figure()
plt.plot(g_losses, label="Generator Loss")
plt.plot(d_losses, label="Discriminator Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("CGAN Training Loss")
plt.legend(); plt.show()

sample_labels = tf.constant(np.arange(10).reshape(-1, 1), dtype=tf.int32)
sample_noise = tf.random.normal((10, 100))
gen_imgs = gen([sample_noise, sample_labels], training=False).numpy()
show_images(gen_imgs, titles=CLASS_NAMES, suptitle="CGAN - One Sample Per Class", n_cols=5)

In [ ]:
def get_inception_features(images):
    imgs = (images + 1) / 2.0
    imgs = tf.image.resize(imgs, (75, 75))
    imgs = tf.keras.applications.inception_v3.preprocess_input(imgs * 255.0)
    return inception_model.predict(imgs, verbose=0)

inception_model = tf.keras.applications.InceptionV3(
    include_top=False, pooling='avg', input_shape=(75, 75, 3), weights='imagenet')

def compute_fid(real_imgs, fake_imgs):
    real_feat = get_inception_features(real_imgs)
    fake_feat = get_inception_features(fake_imgs)
    mu1, sigma1 = real_feat.mean(axis=0), np.cov(real_feat, rowvar=False)
    mu2, sigma2 = fake_feat.mean(axis=0), np.cov(fake_feat, rowvar=False)
    diff = mu1 - mu2
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

real_sample = x_train[np.random.choice(len(x_train), 500, replace=False)]
fake_labels_eval = tf.random.uniform((500, 1), 0, num_classes, dtype=tf.int32)
fake_sample = gen([tf.random.normal((500, 100)), fake_labels_eval], training=False).numpy()
fid_cgan = compute_fid(real_sample, fake_sample)
print("CGAN FID:", fid_cgan)

In [ ]:
gen_uncond = UnconditionalGenerator(latent_dim=100)
disc_uncond = UnconditionalDiscriminator()
g_losses_u, d_losses_u = train_unconditional_gan(gen_uncond, disc_uncond, x_train, latent_dim=100, epochs=10, batch_size=128)

In [ ]:
fake_sample_u = gen_uncond(tf.random.normal((500, 100)), training=False).numpy()
fid_uncond = compute_fid(real_sample, fake_sample_u)

show_images(gen_uncond(tf.random.normal((10, 100)), training=False).numpy(),
            suptitle="Unconditional GAN Samples", n_cols=5)

print("Metric      | Conditional GAN | Unconditional GAN")
print("Final G loss|", g_losses[-1], "|", g_losses_u[-1])
print("Final D loss|", d_losses[-1], "|", d_losses_u[-1])
print("FID         |", fid_cgan, "|", fid_uncond)

In [ ]:
target_classes = ['airplane', 'bird', 'cat', 'ship', 'truck']
class_idx = [CLASS_NAMES.index(c) for c in target_classes]

labels_ctrl = tf.constant(np.array(class_idx).reshape(-1, 1), dtype=tf.int32)
noise_ctrl = tf.random.normal((len(class_idx), 100))
imgs_ctrl = gen([noise_ctrl, labels_ctrl], training=False).numpy()
show_images(imgs_ctrl, titles=target_classes, suptitle="Class-Controlled Generation", n_cols=5)

In [ ]:
airplane_idx = CLASS_NAMES.index('airplane')
labels_fixed = tf.constant([[airplane_idx]] * 5, dtype=tf.int32)
noise_varied = tf.random.normal((5, 100))
imgs_fixed_label = gen([noise_varied, labels_fixed], training=False).numpy()
show_images(imgs_fixed_label, suptitle="Fixed Label (airplane) - Varying Noise", n_cols=5)

In [ ]:
ship_idx = CLASS_NAMES.index('ship')
fixed_noise = tf.random.normal((1, 100))
labels_compare = tf.constant([[airplane_idx], [ship_idx]], dtype=tf.int32)
noise_compare = tf.repeat(fixed_noise, 2, axis=0)
imgs_fixed_noise = gen([noise_compare, labels_compare], training=False).numpy()
show_images(imgs_fixed_noise, titles=['airplane', 'ship'], suptitle="Fixed Noise - Varying Labels", n_cols=2)

In [ ]:
latent_dim_results = {}
for latent_dim in [50, 100, 200]:
    print("\nTraining CGAN with latent_dim =", latent_dim)
    g = Generator(latent_dim=latent_dim, embed_dim=50)
    d = Discriminator(embed_dim=50)
    gl, dl = train_cgan(g, d, x_train, y_train, latent_dim=latent_dim, epochs=10, batch_size=128)
    samples = g([tf.random.normal((100, latent_dim)),
                 tf.random.uniform((100, 1), 0, num_classes, dtype=tf.int32)], training=False).numpy()
    diversity = np.mean(np.std(samples.reshape(100, -1), axis=0))
    sharpness = np.mean(np.abs(np.gradient(samples, axis=1)))
    latent_dim_results[latent_dim] = {"g_loss": gl, "d_loss": dl, "gen": g,
                                       "diversity": diversity, "sharpness": sharpness}

In [ ]:
plt.figure()
for latent_dim, r in latent_dim_results.items():
    plt.plot(r["g_loss"], label=f"latent={latent_dim}")
plt.xlabel("Epoch"); plt.ylabel("Generator Loss")
plt.title("Effect of Latent Dimension - Generator Loss")
plt.legend(); plt.show()

for latent_dim, r in latent_dim_results.items():
    samples = r["gen"]([tf.random.normal((5, latent_dim)),
                         tf.constant([[airplane_idx]] * 5, dtype=tf.int32)], training=False).numpy()
    show_images(samples, suptitle=f"Samples (latent_dim={latent_dim})", n_cols=5)

print("latent_dim | diversity | sharpness | final G loss | final D loss")
for latent_dim, r in latent_dim_results.items():
    print(latent_dim, r["diversity"], r["sharpness"], r["g_loss"][-1], r["d_loss"][-1])

In [ ]:
embed_dim_results = {}
for embed_dim in [25, 50]:
    print("\nTraining CGAN with embedding_dim =", embed_dim)
    g = Generator(latent_dim=100, embed_dim=embed_dim)
    d = Discriminator(embed_dim=embed_dim)
    gl, dl = train_cgan(g, d, x_train, y_train, latent_dim=100, epochs=10, batch_size=128)
    embed_dim_results[embed_dim] = {"g_loss": gl, "d_loss": dl, "gen": g}

In [ ]:
plt.figure()
for embed_dim, r in embed_dim_results.items():
    plt.plot(r["g_loss"], label=f"embed_dim={embed_dim}")
plt.xlabel("Epoch"); plt.ylabel("Generator Loss")
plt.title("Effect of Label Embedding Dimension - Generator Loss")
plt.legend(); plt.show()

for embed_dim, r in embed_dim_results.items():
    samples = r["gen"]([tf.random.normal((5, 100)),
                         tf.constant([[airplane_idx]] * 5, dtype=tf.int32)], training=False).numpy()
    show_images(samples, suptitle=f"Samples (embed_dim={embed_dim})", n_cols=5)

print("embed_dim | final G loss | final D loss")
for embed_dim, r in embed_dim_results.items():
    print(embed_dim, r["g_loss"][-1], r["d_loss"][-1])

In [ ]:
def interpolate_class(gen, class_name, alphas=(0, 0.25, 0.5, 0.75, 1.0), latent_dim=100):
    class_idx = CLASS_NAMES.index(class_name)
    label = tf.constant([[class_idx]], dtype=tf.int32)
    z1 = tf.random.normal((1, latent_dim))
    z2 = tf.random.normal((1, latent_dim))
    imgs = []
    for a in alphas:
        z = (1 - a) * z1 + a * z2
        img = gen([z, label], training=False)[0].numpy()
        imgs.append(img)
    return np.stack(imgs)

for class_name in ['airplane', 'cat', 'ship']:
    imgs = interpolate_class(gen, class_name)
    show_images(imgs, titles=[f"a={a}" for a in (0, 0.25, 0.5, 0.75, 1.0)],
                suptitle=f"Latent Interpolation - {class_name}", n_cols=5)